## Apresentação 2 - Cálculo Numérico
# A PLACA TÉRMICA
**Grupo 1:** Bruno Lopes de Almeida Zuffo, 
João Victor Bozola Bosi, 
Lucas Antero Albuquerque, 
Matheus Marchi Baron, 
Natalia Yumi Watanabe, 
Victor Hugo Albertino e Silva.

---
## 1. Introdução e Parâmetros Iniciais
Antes de iniciarmos a análise dos exercícios, configuramos o ambiente e os parâmetros físicos fixos do problema. Utilizamos a biblioteca `numpy` para álgebra vetorial e `matplotlib` para visualização.

**Parâmetros Nominais:**
* **Dimensões:** $L_x = 2$ cm, $L_y = 1$ cm.
* **Condutividade ($k$):** $0.25$ W/(m·K).
* **Fonte Interna ($f$):** $5.0 \times 10^5$ W/m³.
* **Contornos:** $T_L = 10^\circ C$, $T_R = 30^\circ C$, e gradientes lineares no Topo e Base.

Neste primeiro bloco, importamos todas as bibliotecas matemáticas e os parâmetros operacionais base exigidos no projeto (Seção 2.5).


In [1]:
import matplotlib
matplotlib.use("TkAgg") # Backend necessário para abrir animações em janelas externas
import time
import numpy as np
import matplotlib.pyplot as plt

# Importação das nossas funções de simulação térmica
from functionsT import (ij2n, Assembly, SolveSystem, SolveSystemSparse, 
                        SolveSystemSparse_Circle, PlotaPlaca, Jacobi, 
                        GaussSeidel, AnimacaoTemperatura, 
                        Prepara_Sistema_Otimizado, Resolve_Rapido)

# PARÂMETROS FÍSICOS NOMINAIS
Lx = 0.02  # Comprimento (2 cm)
Ly = 0.01  # Largura (1 cm)
K = 0.25   # Condutividade térmica (W / m.K)

# TEMPERATURAS DE CONTORNO CONSTANTES
TL = 10.0  # Esquerda (°C)
TR = 30.0  # Direita (°C)

fonte = 5.0e5  # Fonte interna de calor (W / m³)


---

## 2. Eficiência Computacional: Matrizes Densas vs Esparsas (Seção 2.5.1 - Exercício 1)

Para resolver a distribuição de calor, dividimos a placa numa malha de $N_x \times N_y$ nós. O modelo clássico gera uma Matriz Densa que cresce ao quadrado ($\mathcal{O}(N^2)$), consumindo rapidamente toda a RAM do computador.

Como a troca térmica ocorre apenas entre vizinhos ortogonais (norte, sul, leste, oeste), nossa função `SolveSystemSparse` armazena apenas as conexões ativas utilizando as matrizes tridiagonais da biblioteca `scipy.sparse`.

Abaixo demonstramos a lógica isolada que gera essa matriz esparsa, seguida da execução comparativa de tempo de máquina e dos gráficos do estado estacionário para malhas progressivamente maiores.


In [ ]:
# DEMONSTRAÇÃO DA LÓGICA ESPARSA (Não executa o cálculo final, apenas ilustra a técnica)
from scipy import sparse

def Demonstracao_Assembly_Esparso(Nx, Ny, k):
    nunk = Nx * Ny
    
    # 1. Criação dos vetores contendo apenas as diagonais principais
    d0 = np.ones(nunk) * 4.0 * k          # Auto-nó (diagonal principal)
    d1 = -np.ones(nunk - 1) * k           # Vizinhos Leste/Oeste
    dN = -np.ones(nunk - Nx) * k          # Vizinhos Norte/Sul
    
    for i in range(1, Ny):
        d1[i * Nx - 1] = 0  # Quebra da conexão artificial entre as extremidades das linhas
    
    # 2. Montagem da matriz esparsa
    A = sparse.diags(
        [dN, d1, d0, d1, dN],
        [-Nx, -1, 0, 1, Nx],
        format='lil'
    )
    return A


In [ ]:
# 2.5.1 - Exercício 1

casos = [(21,11), (41,21), (81,41), (161,81), (321,161)]
resultados = []

for (Nx, Ny) in casos:

    h = Lx / (Nx - 1)
    x_coords = np.linspace(0, Lx, Nx)
    TB = 10 + 20 * (x_coords / Lx) # Gradiente na base
    TT = 10 + 20 * (x_coords / Lx) # Gradiente no topo
    
    # DENSO
    if Nx <= 161: 
        T_d, tA_d, tM_d, tS_d = SolveSystem(Nx, Ny, h, K, TL, TR, TB, TT, fonte)
    else:
        T_d, tA_d, tM_d, tS_d = None, None, None, None
    
    # ESPARSO
    T_s, tA_s, tM_s, tS_s = SolveSystemSparse(Nx, Ny, h, K, TL, TR, TB, TT, fonte)
    
    # TEMPERATURA MÁXIMA
    T_max = np.max(T_s)
    
    resultados.append([
        Nx, Ny,
        tA_d, tM_d, tS_d,
        tA_s, tM_s, tS_s,
        T_max
    ])
    
    # CONTOUR
    PlotaPlaca(Nx, Ny, Lx, Ly, T_s)
    
    # PERFIL CENTRAL (gráfico da temperatura em função do eixo X)
    linha_central = Ny // 2
    perfil = T_s[linha_central, :]
    
    x = np.linspace(0, Lx, Nx)
    
    plt.plot(x, perfil)
    # Título atualizado com o T_max!
    plt.title(f'Eixo Central ({Nx} X {Ny}) | Temp. Máxima: {T_max:.3f} °C')
    plt.xlabel('x (m)')
    plt.ylabel('Temperatura (°C)')
    plt.grid()
    plt.show()

# --- TABELA 1: DESEMPENHO (DENSO vs ESPARSO) ---
print("\n" + "="*90)
print(f"{'COMPARAÇÃO DE TEMPOS (DENSO vs ESPARSO)':^90}")
print("="*90)

header = (
    f"{'Nx':>5} {'Ny':>5} | "
    f"{'Mont.(D)':>10} {'Sist.(D)':>10} {'Resol.(D)':>10} | "
    f"{'Mont.(S)':>10} {'Sist.(S)':>10} {'Resol.(S)':>10}"
)

print(header)
print("-"*90)

for r in resultados:
    print(
        f"{r[0]:5d} {r[1]:5d} | "
        f"{(f'{r[2]:.4f}' if r[2] is not None else '---'):>10} "
        f"{(f'{r[3]:.4f}' if r[3] is not None else '---'):>10} "
        f"{(f'{r[4]:.4f}' if r[4] is not None else '---'):>10} | "
        f"{(f'{r[5]:.4f}' if r[5] is not None else '---'):>10} "
        f"{(f'{r[6]:.4f}' if r[6] is not None else '---'):>10} "
        f"{(f'{r[7]:.4f}' if r[7] is not None else '---'):>10}"
    )
print("="*90)

# --- TABELA 2: CONVERGÊNCIA DA TEMPERATURA MÁXIMA ---
print("\n" + "="*45)
print(f"{'CONVERGÊNCIA DA TEMPERATURA MÁXIMA':^45}")
print("="*45)
print(f"{'Nx':>5} | {'Ny':>5} | {'T_max (°C)':>15}")
print("-" * 45)

for r in resultados:
    print(f"{r[0]:5d} | {r[1]:5d} | {r[8]:15.4f}")
print("="*45)


                         COMPARAÇÃO DE TEMPOS (DENSO vs ESPARSO)                          
   Nx    Ny |   Mont.(D)   Sist.(D)  Resol.(D) |   Mont.(S)   Sist.(S)  Resol.(S)
------------------------------------------------------------------------------------------
   21    11 |     0.0000     0.0000     0.0000 |     0.0000     0.0129     0.0005
   41    21 |     0.0010     0.0034     0.0155 |     0.0072     0.0169     0.0000
   81    41 |     0.0106     0.0396     0.5128 |     0.0022     0.0711     0.0048
  161    81 |     0.0352     0.5060    11.0659 |     0.0128     0.4273     0.0313
  321   161 |        ---        ---        --- |     0.1308     3.7791     0.1557

     CONVERGÊNCIA DA TEMPERATURA MÁXIMA      
   Nx |    Ny |      T_max (°C)
---------------------------------------------
   21 |    11 |         44.7065
   41 |    21 |         44.7727
   81 |    41 |         44.7908
  161 |    81 |         44.7942
  321 |   161 |         44.7957


---

## 3. Seção 2.5.1 - Exercício 2: O Obstáculo Circular 

Agora introduzimos a dificuldade geométrica principal: uma região circular interna de raio $R=0.2$ cm onde a temperatura é forçada a $T_C = 30^\circ C$ (Condição de Dirichlet restrita).
A nossa função `SolveSystemSparse_Circle` iterou espacialmente aplicando uma lógica de distância: se o nó de cálculo caísse dentro da geometria do círculo, sua equação correspondente na matriz $A$ era neutralizada e substituída pela identidade, forçando o valor exato no vetor $b$.


In [ ]:
# 2.5.1 - Exercício 2

# Parâmetros da região circular
R  = 0.002          
xc = 0.75 * Lx     
yc = 0.50 * Ly     
TC = 30.0           

casos_ex2      = [(21, 11), (41, 21), (81, 41), (161, 81), (321, 161)]
resultados_ex2 = []
 
for (Nx, Ny) in casos_ex2:
 
    h        = Lx / (Nx - 1)
    x_coords = np.linspace(0, Lx, Nx)
    TB       = 10 + 20 * (x_coords / Lx)
    TT       = 10 + 20 * (x_coords / Lx)
 
    T_s, tA_s, tM_s, tS_s, circle_mask = SolveSystemSparse_Circle(
        Nx, Ny, h, K, TL, TR, TB, TT, fonte,
        Lx, Ly, R, xc, yc, TC
    )
 
    T_max = T_s.max()
    resultados_ex2.append([Nx, Ny, tA_s, tM_s, tS_s, T_max])
 
    # Curvas de nível 
    x = np.linspace(0, Lx, Nx)
    y = np.linspace(0, Ly, Ny)
    X, Y = np.meshgrid(x, y)
 
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.set_aspect('equal')
    levels = np.linspace(10, 42, 17)
    im  = ax.contourf(X, Y, T_s, levels=levels, cmap='jet')
    ax.contour(X, Y, T_s, levels=levels, linewidths=0.25, colors='k')
    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.22)
    cbar.set_ticks(np.arange(10, 46, 4))
    ax.set(xlabel='x (m)', ylabel='y (m)', title=f'Contours of temperature  ({Nx}×{Ny})')
    ax.set_xticks([0, 0.01, 0.02])
    ax.set_yticks([0.000, 0.005, 0.010])
    plt.tight_layout()
    plt.show()

    
    # Perfil de temperatura ao longo do eixo central
    linha_central = Ny // 2
    perfil        = T_s[linha_central, :]
 
    plt.figure(figsize=(7, 3.5))
    plt.plot(x * 100, perfil, color='crimson', lw=1.8)
    plt.axvline(xc * 100, color='steelblue', ls='--', lw=1.2,
                label=f'Centro do círculo  (x = {xc*100:.1f} cm)')
    plt.title(f'Perfil de temperatura – eixo central  ({Nx}×{Ny})')
    plt.xlabel('x (cm)')
    plt.ylabel('Temperatura (°C)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
 
# Tabela de resultados 
print("\n" + "=" * 90)
print(f"{'RESULTADOS DO EX2 (Região Circular TC = 30°C)':^90}")
print("=" * 90)
 
# Cabeçalho com espaçamentos ajustados e barras separadoras
header2 = (
    f"{'Nx':>6} | {'Ny':>6} | "
    f"{'Mont.(S)':>12} | {'Sist.(S)':>12} | {'Resol.(S)':>12} | "
    f"{'T_max (°C)':>15}"
)
print(header2)
print("-" * 90)
 
# Valores alinhados perfeitamente com o cabeçalho
for r in resultados_ex2:
    print(
        f"{r[0]:6d} | {r[1]:6d} | "
        f"{r[2]:12.4f} | {r[3]:12.4f} | {r[4]:12.4f} | "
        f"{r[5]:15.3f}"
    )
 
print("=" * 90)


                      RESULTADOS DO EX2 (Região Circular TC = 30°C)                       
    Nx |     Ny |     Mont.(S) |     Sist.(S) |    Resol.(S) |      T_max (°C)
------------------------------------------------------------------------------------------
    21 |     11 |       0.0010 |       0.0086 |       0.0029 |          38.265
    41 |     21 |       0.0049 |       0.0285 |       0.0000 |          38.265
    81 |     41 |       0.0080 |       0.1370 |       0.0000 |          38.202
   161 |     81 |       0.1063 |       1.0607 |       0.0231 |          38.144
   321 |    161 |       0.1293 |      16.7347 |       0.1569 |          38.108


---
## 4. Seção 2.5.1 - Exercício 3: 

### Condutividade variável

Neste exercício, a condutividade térmica deixa de ser constante e passa a variar no espaço, sendo definida por:

k(x,y) = 0.2 + 0.05 * sin(3πx/Lx) * sin(3πy/Ly)

---

### Discretização

Para cada nó interno (i,j), a equação discreta é dada por:

(kw + ke + ks + kn) * T(i,j)
- kw * T(i-1,j)
- ke * T(i+1,j)
- ks * T(i,j-1)
- kn * T(i,j+1)
= fonte * h²

onde os coeficientes são obtidos avaliando k(x,y) nas faces:

kw = k(x_i - h/2, y_j)  
ke = k(x_i + h/2, y_j)  
ks = k(x_i, y_j - h/2)  
kn = k(x_i, y_j + h/2)

---

### Implementação

A matriz do sistema passa a ser montada ponto a ponto, pois os coeficientes variam com a posição.

Para cada nó interno:

Ic = ij2n(i, j, Nx)  
Iw = ij2n(i-1, j, Nx)  
Ie = ij2n(i+1, j, Nx)  
Is = ij2n(i, j-1, Nx)  
In = ij2n(i, j+1, Nx)  

A[Ic, Ic] = kw + ke + ks + kn  
A[Ic, Iw] = -kw  
A[Ic, Ie] = -ke  
A[Ic, Is] = -ks  
A[Ic, In] = -kn  

---

In [ ]:
### Resultados

casos_ex3 = [(21,11), (41,21), (81,41), (161,81), (321,161)]  
resultados_ex3 = []

for (Nx, Ny) in casos_ex3:

    h = Lx / (Nx - 1)
    x_coords = np.linspace(0, Lx, Nx)

    TB = 10 + 20 * (x_coords / Lx)
    TT = 10 + 20 * (x_coords / Lx)

    T, tA, tM, tS = SolveSystemSparse_VarK(
        Nx, Ny, h, TL, TR, TB, TT, fonte, Lx, Ly
    )

    T_max = np.max(T)

    resultados_ex3.append([Nx, Ny, tA, tM, tS, T_max])

    PlotaPlaca(Nx, Ny, Lx, Ly, T, flag_type='contour')

    linha_central = Ny // 2
    perfil = T[linha_central, :]

    x = np.linspace(0, Lx, Nx)

    plt.plot(x, perfil)
    plt.title(f'Eixo Central ({Nx}x{Ny}) | Temp. Máxima: {T_max:.3f} °C')
    plt.xlabel('x (m)')
    plt.ylabel('Temperatura (°C)')
    plt.grid()
    plt.show()

---

### Tabela de resultados

print("\n" + "="*90)
print(f"{'RESULTADOS DO EX3 (k variável)':^90}")
print("="*90)
print(f"{'Nx':>5} {'Ny':>5} | {'Mont.(A)':>10} {'Sist.':>10} {'Resol.':>10} | {'T_max':>10}")
print("-"*90)

for r in resultados_ex3:
    print(f"{r[0]:5d} {r[1]:5d} | {r[2]:10.4f} {r[3]:10.4f} {r[4]:10.4f} | {r[5]:10.4f}")

print("="*90)

---
## 5. Seção 2.5.1 - Exercício 4: Estudo Paramétrico da Fonte Circular ($T_C$)

Neste item, analisamos como a temperatura imposta num obstáculo circular ($T_C$) afeta o equilíbrio térmico global da placa. Fisicamente, o círculo atua como uma "torneira térmica" que pode injetar ou remover calor do sistema.

Queremos observar duas grandezas escalares em função de $T_C$:
1. **Temperatura Máxima ($T_{max}$):** O valor absoluto do ponto mais quente da malha.
2. **Temperatura Média ($T_{med}$):** A integral da temperatura sobre a área, aproximada pela média aritmética dos nós: $\bar{T} = \frac{1}{N} \sum_{i=1}^{N} T_i$.

Para isso, executamos um laço que varia $T_C$ de 0 a 100°C. Para cada passo, chamamos a função `SolveSystemSparse_Circle`, que reconstrói o vetor de carga $b$ com a nova condição de contorno interna e resolve o sistema esparso.

In [ ]:
# 2.5.1 - Exercício 4

# Definindo os parâmetros da malha fixa 
Nx_param, Ny_param = 101, 51
h_param = Lx / (Nx_param - 1)

x_coords_param = np.linspace(0, Lx, Nx_param)
TB_param = 10 + 20 * (x_coords_param / Lx)
TT_param = 10 + 20 * (x_coords_param / Lx)

# Vetores de Temperatura do Círculo (TC) para testar
valores_TC = np.linspace(0, 100, 21) 

T_max_list = []
T_med_list = []

# Loop para rodar a simulação para cada valor de TC
for TC_atual in valores_TC:
    
    T_vec, _, _, _, _ = SolveSystemSparse_Circle(
        Nx_param, Ny_param, h_param, K, TL, TR, TB_param, TT_param, fonte, Lx, Ly, R, xc, yc, TC_atual
    )
    
    # O SciPy retorna o vetor 1D. Achamos o máximo e a média com o NumPy
    T_max_list.append(np.max(T_vec))
    T_med_list.append(np.mean(T_vec))

# Geração dos Gráficos
plt.figure(figsize=(8, 5))

# Eixo Y esquerdo (Temperatura Máxima)
ax1 = plt.gca()
linha1, = ax1.plot(valores_TC, T_max_list, color='crimson', marker='o', label='Temp. Máxima')
ax1.set_xlabel('Temperatura do Círculo TC (°C)')
ax1.set_ylabel('Temperatura Máxima da Placa (°C)', color='crimson')
ax1.tick_params(axis='y', labelcolor='crimson')

# Eixo Y direito (Temperatura Média)
ax2 = ax1.twinx() 
linha2, = ax2.plot(valores_TC, T_med_list, color='steelblue', marker='s', label='Temp. Média')
ax2.set_ylabel('Temperatura Média da Placa (°C)', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')

# Título e Legendas
plt.title('Estudo Paramétrico: Efeito de TC na Placa (Malha 101x51)')
ax1.grid(True, linestyle='--', alpha=0.6)

# Juntar as legendas dos dois eixos
linhas = [linha1, linha2]
labels = [l.get_label() for l in linhas]
ax1.legend(linhas, labels, loc='upper left')

plt.tight_layout()
plt.show()

---
## 6. Seção 2.5.1 - Exercício 5: Prova da Linearidade e Relação de Sensibilidade


A equação do calor em regime estacionário ($\nabla^2 T = -f/k$) é uma Equação Diferencial Parcial **linear**. Isso implica que a temperatura em qualquer ponto $T_k$ da placa é uma combinação linear das condições de contorno impostas.


Postulamos que existe uma relação afim para um nó arbitrário $k$:
$$T_k(T_R, T_C) = a \cdot T_R + b \cdot T_C + c$$
Onde:
* $a$ é a sensibilidade de $T_k$ em relação à borda direita.
* $b$ é a sensibilidade em relação ao obstáculo circular.
* $c$ é a contribuição constante (fonte interna e outras bordas).


Para encontrar os coeficientes $(a, b, c)$, simulamos três cenários distintos (Base, Delta $T_R$ e Delta $T_C$) e resolvemos um sistema de equações simples baseado na variação observada no nó escolhido.

In [ ]:
# 2.5.1 - Exercício 5

print("\n" + "=" * 90)
print(f"{'RESULTADOS DO EX5  (Coeficientes a, b e c)':^90}")
print("=" * 90)

# 1. Definir a malha para o Exercício 5 (conforme sugerido no material)
Nx_5, Ny_5 = 101, 51 
h_5 = Lx / (Nx_5 - 1)
x_coords_5 = np.linspace(0, Lx, Nx_5)
TB_5 = 10 + 20 * (x_coords_5 / Lx)
TT_5 = 10 + 20 * (x_coords_5 / Lx)

# 2. Escolher um ponto k interno arbitrário (longe do círculo e das bordas)
# Vamos pegar o meio da altura (Ny//2) e um quarto do comprimento (Nx//4)
i_k = Nx_5 // 4
j_k = Ny_5 // 2

# CENÁRIO 1: TR e TC com valores nominais (30°C)
TR_1, TC_1 = 30.0, 30.0
T_s1, _, _, _, _ = SolveSystemSparse_Circle(
    Nx_5, Ny_5, h_5, K, TL, TR_1, TB_5, TT_5, fonte,
    Lx, Ly, R, xc, yc, TC_1
)
Tk_1 = T_s1[j_k, i_k]

# CENÁRIO 2: Aumentar apenas TR (ex: para 40°C)
TR_2, TC_2 = 40.0, 30.0
T_s2, _, _, _, _ = SolveSystemSparse_Circle(
    Nx_5, Ny_5, h_5, K, TL, TR_2, TB_5, TT_5, fonte,
    Lx, Ly, R, xc, yc, TC_2
)
Tk_2 = T_s2[j_k, i_k]

# CENÁRIO 3: Aumentar apenas TC (ex: para 40°C)
TR_3, TC_3 = 30.0, 40.0
T_s3, _, _, _, _ = SolveSystemSparse_Circle(
    Nx_5, Ny_5, h_5, K, TL, TR_3, TB_5, TT_5, fonte,
    Lx, Ly, R, xc, yc, TC_3
)
Tk_3 = T_s3[j_k, i_k]

# CÁLCULO DOS COEFICIENTES (Equação: Tk = a*TR + b*TC + c)
# a = Variação de Tk / Variação de TR
a = (Tk_2 - Tk_1) / (TR_2 - TR_1)

# b = Variação de Tk / Variação de TC
b = (Tk_3 - Tk_1) / (TC_3 - TC_1)

# c = Isolando a constante no Cenário 1
c = Tk_1 - (a * TR_1) - (b * TC_1)

print(f"Ponto k escolhido: índice [i={i_k}, j={j_k}] (x = {i_k*h_5*100:.2f} cm, y = {j_k*h_5*100:.2f} cm)")
print(f"Temperatura Tk no Cenário Nominal (TR=30, TC=30): {Tk_1:.4f} °C\n")
print("Coeficientes encontrados da relação [Tk = a*TR + b*TC + c]:")
print(f"a = {a:.6f}")
print(f"b = {b:.6f}")
print(f"c = {c:.6f}\n")
print(f"Equação Final: Tk = {a:.4f}*TR + {b:.4f}*TC + {c:.4f}")
print("=" * 90)


                        RESULTADOS DO EX5  (Coeficientes a, b e c)                        
Ponto k escolhido: índice [i=25, j=25] (x = 0.50 cm, y = 0.50 cm)
Temperatura Tk no Cenário Nominal (TR=30, TC=30): 33.4691 °C

Coeficientes encontrados da relação [Tk = a*TR + b*TC + c]:
a = 0.000192
b = 0.069052
c = 31.391726

Equação Final: Tk = 0.0002*TR + 0.0691*TC + 31.3917


---

## 7. Seção 2.6.1 - Exercício 1: Comparação entre Jacobi e Gauss-Seidel

Métodos iterativos resolvem $Ax=b$ partindo de um palpite inicial e refinando-o. 
* **Jacobi:** Atualiza todos os pontos simultaneamente usando apenas valores da iteração anterior. 
* **Gauss-Seidel:** Usa os valores já atualizados na iteração corrente para calcular os próximos pontos.

Sendo $A = D + L + U$ (Diagonal, Inferior, Superior):
* **Jacobi:** $x^{(k+1)} = D^{-1} (b - (L+U)x^{(k)})$
* **Gauss-Seidel:** $x^{(k+1)} = (D+L)^{-1} (b - Ux^{(k)})$

Implementamos as funções `Jacobi` e `GaussSeidel` utilizando matrizes esparsas. Medimos o número de iterações necessárias para atingir diferentes níveis de tolerância ($\epsilon$). Esperamos que o Gauss-Seidel converja em aproximadamente metade das iterações do Jacobi.


In [14]:
# 2.6.1 Exercício 1
casos = [(21,11), (41,21), (81,41), (161,81), (321,161)]
TOLs = [1e-2, 1e-4, 1e-6]
MAXITER = 10000

print("\n" + "="*90)
print(f"{'COMPARAÇÃO DE MÉTODOS (JACOBI vs GAUSS-SEIDEL)':^90}")
print("="*90)

header = (
    f"{'Nx':>5} | {'Ny':>5} | {'TOL':>8} | "
    f"{'Jacobi (s)':>12} | {'Iter (J)':>8} | "
    f"{'GS (s)':>12} | {'Iter (GS)':>9}"
)
print(header)
print("-" * 90)

for (Nx, Ny) in casos:
    
    x_coords = np.linspace(0, Lx, Nx)
    TB = 10 + 20 * (x_coords / Lx)
    TT = 10 + 20 * (x_coords / Lx)
    h = Lx / (Nx - 1)
    
    for tol in TOLs:
        
        Tj, it_j, tj, frame = Jacobi(Nx, Ny, h, K, TL, TR, TB, TT, fonte, tol, MAXITER, animation=False, frame_skip=0)
        Tg, it_g, tg, frame = GaussSeidel(Nx, Ny, h, K, TL, TR, TB, TT, fonte, tol, MAXITER, animation=False, frame_skip=0)
        
        print(
            f"{Nx:5d} | {Ny:5d} | {tol:8.0e} | "
            f"{tj:12.4f} | {it_j:8d} | "
            f"{tg:12.4f} | {it_g:9d}"
        )

print("="*90)


                      COMPARAÇÃO DE MÉTODOS (JACOBI vs GAUSS-SEIDEL)                      
   Nx |    Ny |      TOL |   Jacobi (s) | Iter (J) |       GS (s) | Iter (GS)
------------------------------------------------------------------------------------------
   21 |    11 |    1e-02 |       0.0019 |      168 |       0.0273 |        86
   21 |    11 |    1e-04 |       0.0065 |      316 |       0.0506 |       160
   21 |    11 |    1e-06 |       0.0065 |      464 |       0.0841 |       234
   41 |    21 |    1e-02 |       0.0129 |      494 |       0.1033 |       249
   41 |    21 |    1e-04 |       0.0143 |     1090 |       0.2389 |       548
   41 |    21 |    1e-06 |       0.0270 |     1686 |       0.3376 |       846
   81 |    41 |    1e-02 |       0.0291 |     1255 |       0.3937 |       630
   81 |    41 |    1e-04 |       0.1113 |     3646 |       1.1806 |      1827
   81 |    41 |    1e-06 |       0.1815 |     6034 |       1.7863 |      3021
  161 |    81 |    1e-02 |       0.16

---
## 8. Seção 2.6.1 - Exercício 2: Visualização da Convergência Térmica

A iteração numérica pode ser vista como um processo de difusão artificial. Inicialmente, a placa está "fria" (zeros). A cada passo do algoritmo, a informação das temperaturas de contorno "viaja" para o interior da placa.

Utilizamos a função `AnimacaoTemperatura`. Ela captura "fotos" da placa a cada $N$ iterações (`frame_skip`). Isso permite visualizar como o Gauss-Seidel propaga o calor das bordas e da fonte interna até atingir o estado estacionário.

In [10]:
# 2.6.1 - Exercício 2

Nx = 101
Ny = 51 

Lx = 0.02  # 2 cm
Ly = 0.01  # 1 cm

h = Lx / (Nx - 1)
k = 0.25

TL = 10
TR = 30

# contornos variáveis
x_coords = np.linspace(0, Lx, Nx)
TB = 10 + 20 * (x_coords / Lx)
TT = 10 + 20 * (x_coords / Lx)

fonte = 5.0e5

T, it, _, frames = GaussSeidel(Nx, Ny, h, k, TL, TR, TB, TT, fonte, TOL=1e-6, MAXIT=10000, animation=True, frame_skip = 20)
AnimacaoTemperatura(frames, Nx, Ny, Lx, Ly)

T, it, _, frames = Jacobi(Nx, Ny, h, k, TL, TR, TB, TT, fonte, TOL=1e-6, MAXIT=10000, animation=True, frame_skip = 20)
AnimacaoTemperatura(frames, Nx, Ny, Lx, Ly)

---

## 9. Seção 2.6.1 - Exercício 3: Otimização Computacional e Solução Não-Linear

O desafio final pede o processo inverso ao inicial: descobrir qual é o valor ideal de injeção $T_C$ para que a placa atinja a temperatura limite máxima de $39.5^\circ C$ em seu pico. Como o valor procurado depende das saídas dinâmicas, o problema torna-se **não-linear**, e necessitamos do método iterativo dos gradientes (onde a cada ciclo ajustamos $T_C$ combatendo o erro residual).

Se reconstruíssemos e fatorássemos a geometria dezenas de vezes a cada loop, o algoritmo colapsaria ou demoraria vários minutos na malha gigante. 
A arquitetura do código foi dividida em dois motores:
1. `Prepara_Sistema_Otimizado`: Monta a matriz geométrica gigante apenas UMA vez e calcula a fatoração isolada através de `scipy.sparse.linalg.splu`.
2. `Resolve_Rapido`: Opera dentro do laço `while`, injetando a nova temperatura e resolvendo por simples substituição de equações, de forma instantânea.


In [11]:
# 2.6.1 - Exercício 3

print("\n" + "="*90)
print(f"{'RESULTADOS DO EX3 (Procura do TC ótimo para T_max = 39.5°C)':^90}")
print("="*90)

T_alvo = 39.5     # T* definido no PDF
beta = 0.8        # Valor próximo de 1, como sugerido no PDFs
TOL = 1e-5        # Tolerância para o erro
MAXITER = 100     # Segurança para não entrar em loop infinito

# Testar com várias malhas conforme pedido
casos_ex3 = [(21, 11), (41, 21), (81, 41), (161, 81), (321, 161)]

print(f"{'Nx':>6} | {'Ny':>6} | {'TC Encontrado':>18} | {'T_max Final':>18} | {'Iterações':>11}")
print("-" * 90)

for (Nx, Ny) in casos_ex3:
    h_p = Lx / (Nx - 1)
    x_c = np.linspace(0, Lx, Nx)
    TB_p = 10 + 20 * (x_c / Lx)
    TT_p = 10 + 20 * (x_c / Lx)
    
    # PREPARAÇÃO (Fora do while)
    fatorada, b_base, mask_1d = Prepara_Sistema_Otimizado(
        Nx, Ny, h_p, K, TL, TR, TB_p, TT_p, fonte, Lx, Ly, R, xc, yc
    )
    
    TC_atual, k_iter, erro = 30.0, 0, 1.0
    
    # ITERAÇÃO (Dentro do while)
    while abs(erro) > TOL and k_iter < MAXITER:
        T_vec = Resolve_Rapido(fatorada, b_base, mask_1d, TC_atual)
        T_max = np.max(T_vec)
        erro = T_max - T_alvo
        TC_atual -= beta * erro
        k_iter += 1

    print(f"{Nx:6d} | {Ny:6d} | {TC_atual:15.4f} °C | {T_max:15.4f} °C | {k_iter:11d}")

print("=" * 90)


               RESULTADOS DO EX3 (Procura do TC ótimo para T_max = 39.5°C)                
    Nx |     Ny |      TC Encontrado |        T_max Final |   Iterações
------------------------------------------------------------------------------------------
    21 |     11 |         34.2880 °C |         39.5000 °C |          40
    41 |     21 |         34.2492 °C |         39.5000 °C |          40
    81 |     41 |         34.4154 °C |         39.5000 °C |          42
   161 |     81 |         34.5513 °C |         39.5000 °C |          40
   321 |    161 |         34.6373 °C |         39.5000 °C |          40
